In [1]:
import gc

In [2]:
import numpy as np
import xarray as xr

In [3]:
from data import ds_single, ds_pressure, _load_single_dataset

In [4]:
from dataclasses import dataclass


@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]

In [5]:
def compute_ensemble_spread_bounds(
    mean: xr.DataArray, spread: xr.DataArray, percentile: list[float]
) -> EnsembleSpreadBounds:
    mean = mean.values.flatten()
    spread = spread.values.flatten()

    spread_nonzero = spread[spread > 0.0]

    if len(spread_nonzero) > 0:
        absolute = np.nanquantile(spread_nonzero, [1 - p for p in percentile])
    else:
        absolute = [0.0 for _ in percentile]

    abs_mean = np.abs(mean)
    rel = spread[abs_mean > 0.0] / abs_mean[abs_mean > 0.0]
    rel_nonzero = rel[rel > 0.0]

    if len(rel_nonzero) > 0:
        relative = np.nanquantile(rel_nonzero, [1 - p for p in percentile])
    else:
        relative = [0.0 for _ in percentile]

    amin, amax = np.nanmin(mean), np.nanmax(mean)
    # if we map [amin, amax] to [-1, 1], then fitting a quadratic to
    #  (-1, 0), (x', y'), (1, 0) will reduce to
    # f(x) = a - a*x^2
    #  so we can fit a by using
    # y' = a - a*(x')^2
    # y' = a * (1 - (x')^2)
    # a = (y') / (1 - (x')^2)
    a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin)) ** 2)
    a_nonzero = a[a > 0.0]

    if len(a_nonzero) > 0:
        quadratic = np.nanquantile(a_nonzero, [1 - p for p in percentile])
    else:
        quadratic = [0.0 for _ in percentile]

    return EnsembleSpreadBounds(
        percentile=percentile,
        absolute=absolute,
        relative=relative,
        quadratic=quadratic,
    )

In [6]:
from tqdm import tqdm

In [7]:
with open("ensemble-spread-single.txt", "w") as f:
    for i, key in tqdm(enumerate(sorted(ds_single.mean))):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        print(f"{key}: ", end="", flush=True, file=f)
        print(
            f"{compute_ensemble_spread_bounds(ds_single.mean[key], ds_single.spread[key], [1.0, 0.999, 0.99, 0.95])!r}",
            flush=True,
            file=f,
        )

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: invalid value encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: divide by zero encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: divide by zero encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: invalid value encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: divide by zero encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean -

In [8]:
with open("ensemble-spread-pressure.txt", "w") as f:
    for key in tqdm(sorted(ds_pressure.mean)):
        print(f"{key}:", flush=True, file=f)
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(
                f"{compute_ensemble_spread_bounds(ds_pressure.mean[key].sel(pressure_level=pressure), ds_pressure.spread[key].sel(pressure_level=pressure), [1.0, 0.999, 0.99, 0.95])!r}",
                flush=True,
                file=f,
            )

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: divide by zero encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_61437/944095182.py:29: RuntimeWarning: invalid value encountered in divide
  a = spread / (1.0 - (-1.0 + 2.0 * (mean - amin) / (amax - amin))**2)
100%|██████████████████████████████████████████████████████████| 16/16 [08:59<00:00, 33.72s/it]
